In [ ]:
import json

from src.modeling.evaluation import ModelEvaluator
from src.config import ProjectPath

In [ ]:
# [INFO] 1. SETUP CAU HINH & TIM DUONG DAN
data_name = "adenocarcinoma"
n_features = 50

path = ProjectPath(data_name, n_features)
raw_path = path.raw_path

# =====================================================================
# [FIX CHI TIET] Khai bao doc tiep ten file CSV chinh xac 100% tu anh!
# Khong dung ham tu dong ghep chuoi nua de ne loi Typo va dau gach duoi.
# =====================================================================
sfs_path = path.wrapper_dir / "adenocarcinoma_SFS_selected_1seed_3p_20max_raw.csv"
sklearn_sfs_path = path.wrapper_dir/ "adenocarcinoma_sklearn_SFS__1seed_3p_automax_raw.csv"

# [INFO] TU DONG LUC LOI TIM THU MUC LOG MOI NHAT
sfs_runs_dir = path.results_base_dir / "wrapper" / "raw" / "SFS"
sklearn_runs_dir = path.results_base_dir / "wrapper" / "raw" / "sklearn_SFS"

latest_sfs_run = sorted(sfs_runs_dir.glob("run_*"))[-1]
latest_sklearn_run = sorted(sklearn_runs_dir.glob("run_*"))[-1]

sfs_metrics = latest_sfs_run / "metrics" / "metrics.json"
sklearn_metrics = latest_sklearn_run / "metrics" / "metrics.json"

print(f"[*] Dang lay log SFS tu: {latest_sfs_run.name}")
print(f"[*] Dang lay log Sklearn tu: {latest_sklearn_run.name}")

In [ ]:
# [INFO] 2. DOC FILE JSON LAY METRICS BANG THU VIEN CHUAN (Fix loi Pyright)
with open(sfs_metrics, "r", encoding="utf-8") as f:
    sfs_meta = json.load(f)

with open(sklearn_metrics, "r", encoding="utf-8") as f:
    sklearn_meta = json.load(f)

sfs_time_ms = float(sfs_meta["total_fit_time_ms"])
sfs_n_features = int(sfs_meta["n_features_selected"])

sklearn_time_ms = float(sklearn_meta["total_fit_time_ms"])
sklearn_n_features = int(sklearn_meta["n_features_selected"])

In [ ]:
# [INFO] 3. CHAM DIEM BANG MODEL EVALUATOR
evaluator = ModelEvaluator(
    data_name=data_name,
    valid_method=[], 
    n_features=n_features,
)

# Cham SFS Tu Build
evaluator.evaluate_custom_file(
    file_path=str(sfs_path),
    method_label=f"SFS Custom\n({sfs_n_features} feats, {sfs_time_ms/1000:.1f}s)", 
)

# Cham Sklearn SFS
evaluator.evaluate_custom_file(
    file_path=str(sklearn_sfs_path),
    method_label=f"Sklearn SFS\n({sklearn_n_features} feats, {sklearn_time_ms/1000:.1f}s)",
)

In [ ]:
# [INFO] 4. XUAT REPORT VA VE BIEU DO
result_df = evaluator.generate_report_and_plot(save_to_eval_dir=True)

print("\n[RESULT] BANG TONG KET SO SANH:")
result_df